# S10 Pronóstico de humedad a un minuto
**AireIQ — Eberth Emmanuel Rosales Castillo.**

Pregunta: ¿cuál será la humedad media del próximo minuto, usando solo la lectura agregada del minuto actual?

Se utiliza Gold de S3 y se crean pares solo cuando existen dos minutos consecutivos del mismo dispositivo y tramo. No se atraviesan huecos ni se usa como etiqueta «la siguiente fila» cuando no está exactamente a 60 segundos.

Persistencia predice `humedad(t+1) = humedad(t)`. Se compara contra regresión lineal y Random Forest. Separación temporal 60/20/20; se excluyen pares que cruzan cortes. Se elige por RMSE de validación incluyendo la persistencia y se evalúa al ganador en prueba. No se presupone una mejora.

La entrada del modelo es una sola observación agregada por minuto. No necesita historial en memoria para `transform`. Esto no significa aplicar un modelo entrenado con promedios a lecturas instantáneas crudas: el contrato de inferencia mantiene la misma granularidad.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json, math
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
spark=(SparkSession.builder.appName("AireIQ-S10-Pronostico").master("local[2]")
       .config("spark.sql.session.timeZone","UTC").config("spark.sql.shuffle.partitions","2").getOrCreate())
root=Path('/evidence') if Path('/evidence').exists() else Path('/opt/aireiq/outputs')
sources=sorted(p for p in (root/'s03').iterdir() if (p/'manifest.json').exists() and (p/'gold/_SUCCESS').exists())
assert sources
source=sources[-1]
source_manifest=json.loads((source/'manifest.json').read_text())
out=root/'s10'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
out.mkdir(parents=True,exist_ok=False)
data=spark.read.parquet(str(source/'gold'))
w=Window.partitionBy('dispositivo','tramo').orderBy('minuto')
with_next=(data.withColumn('target_time',F.lead('minuto').over(w))
    .withColumn('label',F.lead('humedad_pct').over(w)))
pairs=with_next.filter(F.col('target_time').cast('long')-F.col('minuto').cast('long')==60).cache()
assert pairs.count()>100
times=[r.minuto for r in pairs.select('minuto').distinct().orderBy('minuto').collect()]
cut1,cut2=times[int(len(times)*.6)],times[int(len(times)*.8)]
train=pairs.filter(F.col('target_time')<F.lit(cut1)).cache()
validation=pairs.filter((F.col('minuto')>=F.lit(cut1))&(F.col('target_time')<F.lit(cut2))).cache()
test=pairs.filter(F.col('minuto')>=F.lit(cut2)).cache()
counts={k:v.count() for k,v in [('train',train),('validation',validation),('test',test)]}
assert min(counts.values())>0
assert train.agg(F.max('target_time')).first()[0]<validation.agg(F.min('minuto')).first()[0]
assert validation.agg(F.max('target_time')).first()[0]<test.agg(F.min('minuto')).first()[0]
print('Minutos Gold:',data.count(),'Pares consecutivos:',pairs.count(),'Bloques:',counts)
print('Pares excluidos en cortes:',pairs.count()-sum(counts.values()))


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/20 04:20:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/20 04:20:11 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/11/20 04:20:11 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/11/20 04:20:11 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/11/20 04:20:11 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
                                                                                

Minutos Gold: 4225 Pares consecutivos: 4113 Bloques: {'train': 2466, 'validation': 822, 'test': 823}
Pares excluidos en cortes: 2


In [2]:
features=['humedad_pct','temperatura_c','presion_hpa','hora_sin','hora_cos']
time_sql="SELECT *, sin(2*PI()*(hour(from_utc_timestamp(minuto,'America/Lima'))+minute(minuto)/60.0)/24) AS hora_sin, cos(2*PI()*(hour(from_utc_timestamp(minuto,'America/Lima'))+minute(minuto)/60.0)/24) AS hora_cos FROM __THIS__"
def model(name):
    if name=='persistencia':
        return Pipeline(stages=[SQLTransformer(statement='SELECT *, humedad_pct AS prediction FROM __THIS__')])
    estimator={
        'lineal':LinearRegression(labelCol='label',regParam=.1,maxIter=100),
        'random_forest':RandomForestRegressor(labelCol='label',numTrees=50,maxDepth=6,seed=42)
    }[name]
    return Pipeline(stages=[SQLTransformer(statement=time_sql),VectorAssembler(inputCols=features,outputCol='features',handleInvalid='error'),estimator])
def metrics(pred):
    result={m:RegressionEvaluator(labelCol='label',predictionCol='prediction',metricName=m).evaluate(pred) for m in ['rmse','mae','r2']}
    assert all(math.isfinite(v) for v in result.values())
    return result
results=[]
for name in ['persistencia','lineal','random_forest']:
    fitted=model(name).fit(train)
    pred=fitted.transform(validation).cache()
    score=metrics(pred); results.append({'modelo':name,**score})
    print(name,score,flush=True);pred.unpersist()
# Desempate a favor de persistencia, evitando complejidad sin mejora.
winner=min(results,key=lambda r:(r['rmse'],r['modelo']!='persistencia'))['modelo']
spark.createDataFrame(results).select('modelo','rmse','mae','r2').orderBy('rmse').show()
print('Elegido por validación:',winner)


persistencia {'rmse': 0.36749002339121933, 'mae': 0.161150908353609, 'r2': 0.9972388841381208}


26/11/20 04:20:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


lineal {'rmse': 0.46020948179958293, 'mae': 0.2896643940387029, 'r2': 0.9956698329723597}
random_forest {'rmse': 3.292483236967725, 'mae': 2.0975270513465256, 'r2': 0.7783638134252541}


[Stage 139:>                                                        (0 + 2) / 2]

+-------------+-------------------+------------------+------------------+
|       modelo|               rmse|               mae|                r2|
+-------------+-------------------+------------------+------------------+
| persistencia|0.36749002339121933| 0.161150908353609|0.9972388841381208|
|       lineal|0.46020948179958293|0.2896643940387029|0.9956698329723597|
|random_forest|  3.292483236967725|2.0975270513465256|0.7783638134252541|
+-------------+-------------------+------------------+------------------+

Elegido por validación: persistencia


## Prueba independiente y persistencia del pipeline
El test permanece fuera de la selección. Si un modelo gana en validación y pierde en prueba, se reporta sin cambiar el ganador retrospectivamente. El pipeline completo se guarda y se recarga antes de usarlo en streaming.


In [3]:
development=train.unionByName(validation).cache()
final_model=model(winner).fit(development)
pred=final_model.transform(test).cache()
test_score=metrics(pred)
baseline_score=metrics(test.withColumn('prediction',F.col('humedad_pct')))
final_model.write().save(str(out/'modelo'))
loaded=PipelineModel.load(str(out/'modelo'))
again=loaded.transform(test).select('dispositivo','minuto',F.col('prediction').alias('restored'))
diff=pred.select('dispositivo','minuto','prediction').join(again,['dispositivo','minuto']).agg(F.max(F.abs(F.col('prediction')-F.col('restored')))).first()[0]
assert diff is not None and diff<=1e-9
test.select('dispositivo','minuto','target_time','humedad_pct','temperatura_c','presion_hpa','label').write.parquet(str(out/'test_pairs'))
pred.select('dispositivo','minuto','target_time','humedad_pct','label','prediction').write.parquet(str(out/'predicciones_test'))
relative=100*(baseline_score['rmse']-test_score['rmse'])/baseline_score['rmse'] if baseline_score['rmse'] else None
report=dict(status='trained_and_verified',s03_run=source.name,source_sha256=source_manifest['source_sha256'],
    run_id=out.name,objetivo='humedad media del siguiente minuto',horizonte_segundos=60,
    unidad_entrada='promedio por minuto',spark_version=spark.version,timezone='UTC; hora ciclica America/Lima',
    total_minutos=data.count(),pares_consecutivos=pairs.count(),conteos=counts,
    pares_cruzando_cortes_excluidos=pairs.count()-sum(counts.values()),features=features,
    validacion=results,modelo_elegido=winner,test=test_score,baseline_test=baseline_score,
    mejora_rmse_test_porcentaje=relative,diferencia_recarga=diff,
    test_start=str(test.agg(F.min('minuto')).first()[0]),test_end=str(test.agg(F.max('target_time')).first()[0]))
(out/'manifest.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(report,indent=2,ensure_ascii=False))
print('Modelo y evidencia:',out)
for frame in [pairs,train,validation,test,development,pred]:frame.unpersist()


{
  "status": "trained_and_verified",
  "s03_run": "20261120T030852967375Z",
  "source_sha256": "35fe207bedce69285c7eae033d7bdc4bccc3dfc6bf4055c231b7a8a2eb4583c4",
  "run_id": "20261120T042015759846Z",
  "objetivo": "humedad media del siguiente minuto",
  "horizonte_segundos": 60,
  "unidad_entrada": "promedio por minuto",
  "spark_version": "4.2.0",
  "timezone": "UTC; hora ciclica America/Lima",
  "total_minutos": 4225,
  "pares_consecutivos": 4113,
  "conteos": {
    "train": 2466,
    "validation": 822,
    "test": 823
  },
  "pares_cruzando_cortes_excluidos": 2,
  "features": [
    "humedad_pct",
    "temperatura_c",
    "presion_hpa",
    "hora_sin",
    "hora_cos"
  ],
  "validacion": [
    {
      "modelo": "persistencia",
      "rmse": 0.36749002339121933,
      "mae": 0.161150908353609,
      "r2": 0.9972388841381208
    },
    {
      "modelo": "lineal",
      "rmse": 0.46020948179958293,
      "mae": 0.2896643940387029,
      "r2": 0.9956698329723597
    },
    {
      "mod

## Límites de la evaluación
RMSE y MAE se expresan en puntos porcentuales. La prueba no representa otros ambientes o dispositivos. La reproducción de Kafka utilizará observaciones históricas de prueba, con marca de replay; no es pronóstico del sensor en vivo. El error contra la etiqueta histórica es verificación retrospectiva; medir error al llegar una observación futura en operación corresponde a la integración posterior.

La ruta de inferencia no debe esperar las ventanas finalizadas por watermark de S8: para un horizonte de un minuto, ese retraso de dos minutos haría llegar tarde el pronóstico. Se mantiene un contrato de minutos cerrados independiente del sink de ventanas tardías. El adaptador continuo de minutos sigue pendiente.
